# EUR/USD volatility with path signatures: new FX channels

A cleaned, FX-only version of `Mihika/volatility_experiments.ipynb`. The question is the same: do
path signatures of the recent hourly path help forecast volatility, beyond HAR and beyond
signature-free features built from the *same* window? What is new:

- **Channels built for FX**, from the hourly bars (`forecast_data.fx_hourly_channels`, past data only):
  - `price_sa`: log price rebuilt from **time-of-day-adjusted returns**. Each hour's return is divided by
    how volatile that hour of the week (New York time) usually is, so the path shows unusual moves, not
    the clock (Asia / London / New York sessions, the 17:00 rollover).
  - `range_sa`: the hourly high-low range, adjusted the same way.
  - `spread`: the **abnormal bid-ask spread**, log spread minus its usual level for that hour (0 = normal,
    above 0 = wider than usual). Liquidity drying up often comes before volatility.
  - `activity`: Dukascopy volume, as before.
- **Two targets**: volatility (as before) and **volatility direction**: will volatility over the next
  h days be higher than over the last h days?
- **One horizon set**: 1 and 5 trading days, with daily-bar targets and baselines and signatures from the
  hourly path (the `*_i` horizons of the old notebook).

What was removed from the old notebook, and why:

| Removed | Why |
|---|---|
| Daily-path signature models | A 22-bar daily path is too short to have much shape; all new channels are hourly |
| 22-day horizon | About 160 non-overlapping windows since 2012: too few to judge |
| HMM anomaly flag section | Added nothing; replaced by the anomaly study |
| Signature-settings sweep (5 extra runs) | Overlaps the ablations; depth 3 + lead-lag was at or near the best on crypto |
| GARCH, LSTM, XGBoost/MLP on stats | GARCH loses to HAR; the LSTM/MLP tie Ridge at much higher cost |
| Drawdown section, calendar dummies, tick counts | Not part of this round |

**Order:** sections 1-6 and 8 run on their own. Section 7 (anomaly score in the forecasts) needs
`anomaly_fx.ipynb` to have run first.

In [ ]:
import logging
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import fx_lab as L
from fx_lab import ex, rp

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S", force=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

QUICK = False  # True: data from 2021 and few epochs, to check that everything runs (results in *_quick)
print("EUR/USD hourly bars through", L.END, "(end of the last finished month)")

## 1. Data

Hourly EUR/USD mid prices with the bid-ask spread, from Dukascopy's BID and ASK hourly candle files
(two downloads per month, cached in `Mihika/data/dukascopy/*_1h_ba_*`). The first run downloads about
360 files and Dukascopy rate-limits, so it can take an hour; after that it is instant. Runs stop at the
end of the last finished month, so the slow tick-file fallback for the current month is never used.

Check that the adjustment does what it should: after it, the |return| and range lines are roughly flat
across the day, and the abnormal spread is centred on zero at every hour.

In [ ]:
hourly = L.load_hourly(start="2021-01-01" if QUICK else L.START)
print(f"{len(hourly):,} hourly bars, {hourly.index[0]} to {hourly.index[-1]}; "
      f"median spread {1e4 * hourly.spread.median():.2f} bp")
L.plot_seasonality(hourly); plt.show()

## 2. Run the volatility forecasts

Fixed in advance (not tuned here): depth-3 signatures with lead-lag and time, the hourly path over the
last 22 trading days, channels `price_sa, activity, range_sa, spread`, 3 expanding purged folds.

| Model | What it is |
|---|---|
| Mean, HAR-RV-L | Benchmarks (HAR on daily realized variance with a leverage term) |
| HAR-RV-L calibrated | HAR with its average bias removed; anything that only fixes HAR's level ties with it |
| HAR-RV-L (intraday RV) | HAR on hourly realized variance: the strong baseline |
| Ridge \| stats | Ridge on summary statistics of the daily window |
| Ridge \| intraday signature | Ridge on the signature of the hourly path |
| Ridge \| intraday lag bank (matched) | Same increments as flat lagged terms, exactly as many columns: is it the signature, or the information? |
| Ridge \| intraday multiscale stats | Hand-built realized-variance features at several scales from the same window |
| Ridge \| intraday signature + HAR-RV | Does the signature add anything on top of the strong baseline? |
| MLP \| intraday signature / lag bank | The same comparison with a nonlinear learner |

Ablations (Ridge on the hourly signature, same rows and folds) run in the same pass: signature levels,
augmentation, and the channel settings in `L.CHANNEL_ABLATIONS`. Finished horizons are skipped on re-run.

In [ ]:
cfg = L.vol_config(quick=QUICK)
ex.run_all(cfg, subprocess_per_horizon=True)
res = rp.load_results(cfg.output_dir)
pd.DataFrame(res.meta).T[["n_rows", "horizon_rows", "n_folds", "first_time", "last_time", "seconds"]]

## 3. Ranking

Fold-mean RMSE; `Pct_vs_HAR` > 0 = better than HAR-RV-L. Then pooled out-of-sample RMSE with
Diebold-Mariano tests against the best model and against HAR-RV-L (lag = the horizon in rows).

In [ ]:
summ = rp.summary_table(res)
display(summ.style.format({"Mean_RMSE": "{:.6g}", "Fold_SD": "{:.6g}", "Pct_vs_Mean": "{:.1f}", "Pct_vs_HAR": "{:.1f}"}))
for hk in res.meta:
    print(ex.HORIZON_LABELS[hk])
    display(rp.dm_table(res, hk).style.format(precision=4))

## 4. Is it the signature?

Each model against what it must beat, on identical rows. `Pct_better` > 0 = lower RMSE; a small `DM_p`
means the difference is significant. In the figure, green / red = significantly better / worse, grey =
the data cannot tell them apart. The signature has to beat the **matched lag bank** and the
**multiscale stats** for its geometry to matter, and **HAR on intraday RV** to be useful at all.

In [ ]:
display(rp.pair_table(res, L.VOL_COMPARISONS).style.format(precision=4))
rp.plot_pair_skill(res, rp.as_pairs(L.VOL_COMPARISONS)); plt.show()

**Calibration and stability.** Are the forecasts the right size (points on the diagonal), and is any
gain steady over time or from a few episodes (rolling RMSE relative to HAR-RV-L; below 1 = better)?

In [ ]:
KEY = ["Ridge | intraday signature", "Ridge | intraday signature + HAR-RV", "HAR-RV-L calibrated", "HAR-RV-L (intraday RV)"]
rp.plot_calibration(res, models=("HAR-RV-L", "HAR-RV-L calibrated", "HAR-RV-L (intraday RV)", "Ridge | intraday signature")); plt.show()
rp.plot_error_over_time(res, KEY); plt.show()

## 5. What do the new channels add?

Ridge on the hourly signature under each setting, same rows and folds. `Pct_vs_group_ref` compares
with the first setting of each group: level 1 only, the raw path, and the **original channels** (price,
activity, range: what the previous FX run used). Read the channel rows as:

- *Adjusted, no spread* vs *Original*: the effect of the time-of-day adjustment;
- *Adjusted + spread* vs *Adjusted, no spread*: the effect of the spread channel.

These are descriptive (the main setting was fixed before seeing them). A difference of a few percent is
within noise.

In [ ]:
abl = L.ablation_table(res)
display(abl.style.format({"Mean_RMSE": "{:.6g}", "Pct_vs_HAR": "{:.1f}", "Pct_vs_group_ref": "{:.1f}"}))

## 6. Volatility direction

**Up** = realized volatility over the next h days is above the last h days (about half the rows). Two
ways to predict it, scored on the same out-of-sample rows:

- **From the volatility forecasts** above, with no refit: a forecast above the past level means "up",
  and log(forecast / past level) ranks the rows.
- **Logistic classifiers** fit on the same purged folds: HAR features alone (the reference), and HAR plus
  one representation of the hourly window (intraday HAR-RV, multiscale stats, the matched lag bank, or
  the signature).

`AUC` ranks rows (0.5 = coin flip); `Hit_rate` is the share called correctly. `AUC_vs_ref` is the AUC
minus that of the HAR classifier, with a 90% block-bootstrap interval; `p_one_sided` small = reliably
better. The signature is useful here only if *HAR + signature* beats *HAR + lag bank* and *HAR + intraday
HAR-RV*, not just HAR alone.

In [ ]:
droot = L.run_direction(cfg)
ddf, dmeta = L.load_direction(droot)
dtab = L.direction_table(ddf, dmeta)
display(dtab.style.format(precision=3))
L.plot_direction(dtab); plt.show()

## 7. The anomaly score in the forecasts

Needs `anomaly_fx.ipynb` first: it selects the best signature detector and the best signature-free
detector for each horizon. Each detector's score (its percentile among the training rows) is used
two ways, refit per fold:

- as an extra input to HAR (`HAR-RV-L + anomaly`);
- as a correction of HAR's error (`HAR-RV-L + anomaly residual`).

The decisive comparison is against **HAR-RV-L calibrated**: on crypto, most of the apparent gain from
the anomaly score was HAR's average bias, which a plain rescaling removes. `Pct_sig_vs_free` > 0 means
the signature detector's version is better than the signature-free one's.

In [ ]:
acfg = L.anomaly_config(quick=QUICK)
selection = Path(acfg.output_dir) / "best_configs.json"
if not selection.exists():
    print(f"Run anomaly_fx.ipynb first (no {selection})")
else:
    vcfg = replace(cfg, use_anomaly=True, anomaly_selection=str(selection), run_ablations=False,
                   models=("Mean", "HAR-RV-L", "HAR-RV-L calibrated", *L.ANOMALY_MODELS),
                   output_dir=str(L.RESULTS / ("vol_anomaly_quick" if QUICK else "vol_anomaly")))
    ex.run_all(vcfg, subprocess_per_horizon=True)
    ares = rp.load_results(vcfg.output_dir)
    display(rp.anomaly_effect_table(ares).style.format(precision=3))
    rp.plot_anomaly_response(ares); plt.show()
    rp.plot_anomaly_timeline(ares); plt.show()

## 8. Compared with the previous FX run

The previous run (`Mihika/results/fx/intraday`, original channels) next to this one, for the models
both have. Its rows end on the day it ran, so compare levels, not row by row: the like-for-like
channel comparison is the ablation table in section 5.

In [ ]:
prev = L.compare_with_previous(res)
display(prev.style.format(precision=4) if prev is not None else "No previous FX intraday run found.")

## 9. Save the summary tables

In [ ]:
out = L.RESULTS / ("summary_quick" if QUICK else "summary")
out.mkdir(parents=True, exist_ok=True)
summ.to_csv(out / "vol_summary.csv", index=False)
abl.to_csv(out / "vol_ablations.csv", index=False)
dtab.to_csv(out / "vol_direction.csv", index=False)
rp.pair_table(res, L.VOL_COMPARISONS).to_csv(out / "vol_comparisons.csv", index=False)
print("saved to", out)